In [1]:
"""
Driving Stress Calculation
Authors: Hannah Picton, Dan Goldberg, Andrew Tedstone 
Revised: 9th November 2026
Description: Using Equation 3 of the associated manuscript, this script calculates the relavant contribution of changes in ice thickness (H) and ice surface slope (S) to the increase in driving stress, and consequent ice flow acceleration, observed at LEV6 between 2009 and 2021. A conservative estimate of uncertainty in the height component (±0.25 m) is also propogated. 
"""

# Import libraries 
import math
import numpy as np

# Ellipsoidal heights (IGSR3) and pole-corrected heights are from data/PPP_outputs/site_coords_IGSR3_ellps_polecorrected.csv
# (output of process_site_positions.py; values given here to 2 d.p.). Geoid corrections and bed elevations are extracted from BedMachine v5.

# LEV5
# -- 2009 -- #
lev5_2009_ellipsoidal_height = 1482.78
lev5_2009_ellip_height_pole_corrected = 1481.71
lev5_geoid_correction = 37                                                                                      # Extracted from Bedmachine v5
lev5_2009_bed = 243.05                                                                                          # Extracted from Bedmachine v5
lev5_2009_height_geoid = lev5_2009_ellip_height_pole_corrected - lev5_geoid_correction
lev5_2009_thickness = lev5_2009_height_geoid - lev5_2009_bed
# -- 2021 -- #
lev5_2021_ellipsoidal_height = 1477.35
lev5_2021_ellip_height_pole_corrected = 1475.75
lev5_2021_height_geoid = lev5_2021_ellip_height_pole_corrected - lev5_geoid_correction
lev5_2021_thickness = lev5_2021_height_geoid - lev5_2009_bed 

# LEV6 
# -- 2009 -- #
lev6_2009_ellipsoidal_height = 1714.49
lev6_2009_ellip_height_pole_corrected = 1713.49
lev6_geoid_correction = 39                                                                                      # Extracted from Bedmachine v5
lev6_2009_bed = 234.31                                                                                          # Extracted from Bedmachine v5
lev6_2009_height_geoid = lev6_2009_ellip_height_pole_corrected - lev6_geoid_correction
lev6_2009_thickness = lev6_2009_height_geoid - lev6_2009_bed
# -- 2021 -- #
lev6_2021_ellipsoidal_height = 1713.33
lev6_2021_ellip_height_pole_corrected = 1710.73
lev6_2021_height_geoid = lev6_2021_ellip_height_pole_corrected - lev6_geoid_correction
lev6_2021_thickness = lev6_2021_height_geoid - lev6_2009_bed 

# Distance between LEV5 and LEV6 from data/PPP_outputs/PPP_lev5_lev6_baselines_IGSR3.csv (output of process_site_positions.py)
# Note: slopes are calculated here from geoid-referenced heights, rather than taken from PPP_lev5_lev6_baseline_slopes_IGSR3.csv (ellipsoidal heights)
dist_2009 = 35961.199784999364
dist_2021 = 35985.817838664356

# Calculate slope between LEV5 and LEV6 in 2009 and 2021
slope_2009 = (lev6_2009_height_geoid - lev5_2009_height_geoid) / dist_2009
slope_2021 = (lev6_2021_height_geoid - lev5_2021_height_geoid) / dist_2021

# Calculate the change in slope and thickness between 2009 and 2021 at LEV6
delta_s = slope_2021 - slope_2009
delta_thickness = lev6_2021_thickness - lev6_2009_thickness

# Calculate fractional changes for response equations
X_S = delta_s / slope_2009
X_T = delta_thickness / lev6_2009_thickness

# --- Print Baseline Summary ---
print("=== BASELINE HEIGHT & THICKNESS ===")
print(f"LEV5 2009 Geoid Height: {lev5_2009_height_geoid:.1f} m | Thickness: {lev5_2009_thickness:.1f} m")
print(f"LEV5 2021 Geoid Height: {lev5_2021_height_geoid:.1f} m | Thickness: {lev5_2021_thickness:.1f} m")
print(f"LEV6 2009 Geoid Height: {lev6_2009_height_geoid:.1f} m | Thickness: {lev6_2009_thickness:.1f} m")
print(f"LEV6 2021 Geoid Height: {lev6_2021_height_geoid:.1f} m | Thickness: {lev6_2021_thickness:.1f} m")
print("\n=== LEV6 SLOPE & THICKNESS CHANGE ===")
print(f"LEV6 2009 Slope: {slope_2009:.6f} ({math.degrees(math.atan(slope_2009)):.4f}°)")
print(f"LEV6 2021 Slope: {slope_2021:.6f} ({math.degrees(math.atan(slope_2021)):.4f}°)")
print(f"LEV6 ΔS:         {delta_s:.6f} ({X_S * 100:.1f}%)")
print(f"LEV6 ΔH:         {delta_thickness:.2f} m ({X_T * 100:.1f}%)\n")

=== BASELINE HEIGHT & THICKNESS ===
LEV5 2009 Geoid Height: 1444.7 m | Thickness: 1201.7 m
LEV5 2021 Geoid Height: 1438.8 m | Thickness: 1195.7 m
LEV6 2009 Geoid Height: 1674.5 m | Thickness: 1440.2 m
LEV6 2021 Geoid Height: 1671.7 m | Thickness: 1437.4 m

=== LEV6 SLOPE & THICKNESS CHANGE ===
LEV6 2009 Slope: 0.006390 (0.3661°)
LEV6 2021 Slope: 0.006474 (0.3709°)
LEV6 ΔS:         0.000085 (1.3%)
LEV6 ΔH:         -2.76 m (-0.2%)



In [2]:
# --- UNCERTAINTY PROPAGATION --- #
uncertainty_H = 0.25     # Height uncertainty in meters

uncertainty_H_diff = math.sqrt(uncertainty_H**2 + uncertainty_H**2)
delta_slope_2009 = abs(slope_2009) * (uncertainty_H_diff / (lev6_2009_height_geoid - lev5_2009_height_geoid))
delta_slope_2021 = abs(slope_2021) * (uncertainty_H_diff / (lev6_2021_height_geoid - lev5_2021_height_geoid))
delta_delta_s = math.sqrt(delta_slope_2009**2 + delta_slope_2021**2)

delta_T = math.sqrt(uncertainty_H**2 + uncertainty_H**2)
delta_delta_T = math.sqrt(delta_T**2 + delta_T**2)

fractional_uncertainty_slope = math.sqrt((delta_delta_s / abs(delta_s))**2 + (delta_slope_2009 / abs(slope_2009))**2)
delta_X_S = abs(X_S) * fractional_uncertainty_slope
fractional_uncertainty_thickness = math.sqrt((delta_delta_T / abs(delta_thickness))**2 + (uncertainty_H / lev6_2009_thickness)**2)
delta_X_T = abs(X_T) * fractional_uncertainty_thickness  

print(f'Uncertainty of {uncertainty_H} m propagated to slope and thickness changes.')

Uncertainty of 0.25 m propagated to slope and thickness changes.


In [3]:
# --- BACK-CALCULATE SLOPE ANGLE ENVELOPES --- #
print("=== BACK-CALCULATED SLOPE ANGLE ENVELOPE ===")
print("-" * 50)
def print_angle(label, slope, delta_slope):
    angle = math.degrees(math.atan(slope))
    delta_angle = (1 / (1 + slope**2)) * delta_slope * (180 / math.pi)
    print(f"{label} Slope Angle:")
    print(f"  Nominal:     {angle:.6f}°")
    print(f"  Uncertainty: +/- {delta_angle:.6f}°")
    print(f"  Range:       ({angle - delta_angle:.6f} to {angle + delta_angle:.6f})°\n")

print_angle("2009", slope_2009, delta_slope_2009)
print_angle("2021", slope_2021, delta_slope_2021)

=== BACK-CALCULATED SLOPE ANGLE ENVELOPE ===
--------------------------------------------------
2009 Slope Angle:
  Nominal:     0.366096°
  Uncertainty: +/- 0.000563°
  Range:       (0.365533 to 0.366659)°

2021 Slope Angle:
  Nominal:     0.370940°
  Uncertainty: +/- 0.000563°
  Range:       (0.370377 to 0.371503)°



In [4]:
# --- RESPONSE EQUATIONS --- #
init_vel = 62.6 # Initial velocity at LEV6 in 2009 (m/yr)
n = 3 # Cuffey & Paterson (2010)
observed_delta_v = 3.2  # Observed change in annual ice velocity at LEV6 between 2009 and 2021 (m/yr)

def deformation_response(dS, dH, err_dS, err_dH, n, U0):
    # dS: relative change in slope
    # dH: relative change in thickness
    # err_dS, err_dH: errors as calculated above
    # n: glen's exponent
    # U0: initial speed
    delU = U0 * ((n + 1) * dH + n * dS)
    err_delU = U0 * math.sqrt(((n + 1) * err_dH)**2 + (n * err_dS)**2)
    return delU, err_delU

def sliding_response(dS, dH, err_dS, err_dH, p, ut, U0):
    # dS: relative change in slope
    # dH: relative change in thickness
    # err_dS, err_dH: errors as calculated above
    # p: sliding exponent
    # U0: initial speed
    # ut: transition speed
    delU = U0 * (1 + U0 / ut) * (p * dH + p * dS)
    err_delU = U0 * (1 + U0 / ut) * math.sqrt((p * err_dH)**2 + (p * err_dS)**2)
    return delU, err_delU

def generalised_response(dS, dH, err_dS, err_dH, p, ud_frac, ut):
    # dS: relative change in slope
    # dH: relative change in thickness
    # err_dS, err_dH: errors as calculated above
    # p: exponent in reg. coulomb sliding law
    # ud_frac: fraction of deformation in total velocity
    # ut: transition speed
    Ud = ud_frac * init_vel
    Ub = init_vel - Ud
    delUb, err_delUb = sliding_response(dS, dH, err_dS, err_dH, p, ut, Ub)
    delUd, err_delUd = deformation_response(dS, dH, err_dS, err_dH, n, Ud)
    delU = delUb + delUd
    err_delU = math.sqrt(err_delUb**2 + err_delUd**2)
    return delU, err_delU

# --- PRINT RESULTS --- #
print("=== PURE INTERNAL DEFORMATION CASE ===")
delU_def, err_def = generalised_response(X_S, X_T, delta_X_S, delta_X_T, p=3, ud_frac=1.0, ut=300)
pct_def = (delU_def / observed_delta_v) * 100
err_pct_def = (err_def / observed_delta_v) * 100

# Get slope-only and thickness-only contributions for the breakdown display
slope_only_def, err_s_def = deformation_response(X_S, 0, delta_X_S, 0, n, init_vel)
thick_only_def, err_t_def = deformation_response(0, X_T, 0, delta_X_T, n, init_vel)
slope_pct_def = (slope_only_def / observed_delta_v) * 100
err_s_pct_def = (err_s_def / observed_delta_v) * 100
thick_pct_def = (thick_only_def / observed_delta_v) * 100
err_t_pct_def = (err_t_def / observed_delta_v) * 100

print(f"Contribution: {delU_def:.2f} ± {err_def:.2f} m")
print(f"Percentage of observed ({observed_delta_v} m): {pct_def:.1f}% ± {err_pct_def:.1f}%")
print(f"  -> Slope contribution:        {slope_pct_def:.1f}% ± {err_s_pct_def:.1f}%")
print(f"  -> Thickness contribution:    {thick_pct_def:.1f}% ± {err_t_pct_def:.1f}%\n")


# --- 2. Sensitivity Tests across Basal Fractions ---
def run_sensitivity_test(p_val, ut_val, label):
    print(f"=== SENSITIVITY TEST: {label} (p={p_val}, ut={ut_val}) ===")
    print(f"{'Basal Fraction':<15} | {'Contribution (m)':<18} | {'% of Observed':<15} | {'Slope contribution % ':<18} | {'Thickness contribution % '}")
    print("-" * 105)

    basal_fractions = [0.60, 0.70, 0.80, 0.90, 0.96]

    for bf in basal_fractions:
        ud_f = 1.0 - bf  
        delU, err_delU = generalised_response(X_S, X_T, delta_X_S, delta_X_T, p_val, ud_f, ut_val)
        pct = (delU / observed_delta_v) * 100
        err_pct = (err_delU / observed_delta_v) * 100
        
        # Isolate slope and thickness components for this basal fraction
        Ud = ud_f * init_vel
        Ub = init_vel - Ud
        
        ds_val, err_ds = sliding_response(X_S, 0, delta_X_S, 0, p_val, ut_val, Ub)
        dt_val, err_dt = sliding_response(0, X_T, 0, delta_X_T, p_val, ut_val, Ub)
        
        if Ud > 0:
            dds_val, err_dds = deformation_response(X_S, 0, delta_X_S, 0, n, Ud)
            ddt_val, err_ddt = deformation_response(0, X_T, 0, delta_X_T, n, Ud)
            tot_s = ds_val + dds_val
            tot_t = dt_val + ddt_val
            err_s_tot = math.sqrt(err_ds**2 + err_dds**2)
            err_t_tot = math.sqrt(err_dt**2 + err_ddt**2)
        else:
            tot_s, tot_t = ds_val, dt_val
            err_s_tot, err_t_tot = err_ds, err_dt

        slope_pct = (tot_s / observed_delta_v) * 100
        err_s_pct = (err_s_tot / observed_delta_v) * 100
        thick_pct = (tot_t / observed_delta_v) * 100
        err_t_pct = (err_t_tot / observed_delta_v) * 100

        print(
            f"{int(bf*100)}% basal       | "
            f"{delU:5.2f} ± {err_delU:4.2f}       | "
            f"{pct:5.1f} ± {err_pct:4.1f}    | "
            f"{slope_pct:5.1f} ± {err_s_pct:4.1f}          | "
            f"{thick_pct:5.1f} ± {err_t_pct:4.1f}"
        )
    print()

run_sensitivity_test(p_val=3, ut_val=300, label="Hard-bed Configuration")
run_sensitivity_test(p_val=5, ut_val=200, label="Soft-bed Configuration")

=== PURE INTERNAL DEFORMATION CASE ===
Contribution: 2.01 ± 0.42 m
Percentage of observed (3.2 m): 62.7% ± 13.1%
  -> Slope contribution:        77.7% ± 12.8%
  -> Thickness contribution:    -15.0% ± 2.7%

=== SENSITIVITY TEST: Hard-bed Configuration (p=3, ut=300) ===
Basal Fraction  | Contribution (m)   | % of Observed   | Slope contribution %  | Thickness contribution % 
---------------------------------------------------------------------------------------------------------
60% basal       |  2.24 ± 0.33       |  69.9 ± 10.2    |  83.5 ± 10.0          | -13.6 ±  1.8
70% basal       |  2.31 ± 0.35       |  72.1 ± 11.1    |  85.6 ± 10.9          | -13.5 ±  1.8
80% basal       |  2.39 ± 0.40       |  74.5 ± 12.3    |  88.0 ± 12.2          | -13.5 ±  2.0
90% basal       |  2.47 ± 0.44       |  77.3 ± 13.9    |  90.8 ± 13.7          | -13.5 ±  2.2
96% basal       |  2.53 ± 0.48       |  79.0 ± 14.9    |  92.6 ± 14.7          | -13.6 ±  2.4

=== SENSITIVITY TEST: Soft-bed Configuration (p